In [1]:
import random

from sldl import SignLanguageDataset
from sldl.configs import LSFBIsolConfig

from sign_language_tools.player import VideoPlayer
from sign_language_tools.pose.mediapipe.edges import UPPER_POSE_EDGES, HAND_EDGES, LIPS_EDGES, EYE_EDGES

In [2]:
root = "F:/datasets/sign-language/lsfb-isol"

# dataset = SignLanguageDataset(
#     shards_url=f"file:{root}/shards/500/" + "shard_000000.tar",
#     body_parts=('upper_pose', 'left_hand', 'right_hand', 'lips', 'left_eye', 'right_eye'),
#     isolated=True,
#     annotations=None,
#     show_loading_progress=True,
#     load_videos=True,
#     video_path="F:/datasets/sign-language/lsfb-isol/videos.tar",
#     video_index_path=f"F:/datasets/sign-language/lsfb-isol/videos.tar.index.json",
# )
dataset = SignLanguageDataset.from_config(LSFBIsolConfig(
    root=root,
    variant='500',
    split='testing',
))
len(dataset)

Loading dataset [file:F:/datasets/sign-language/lsfb-isol/shards/500/shard_000000.tar]...


Loading samples: 9692 samples [00:08, 1205.28 samples/s]

Loaded 9692 samples.


9692

In [3]:
example_sample = dataset[0]
print("Sample ID:", example_sample['id'])
print("Label:", example_sample['label'])
print("Signer:", example_sample['signer-id'])
print("Nb. of frames:", example_sample['n_frames'])
print("Pose sequence - body parts:", set(example_sample['poses'].keys()))
print("Pose sequence - upper pose shape:", example_sample['poses']['upper_pose'].shape)
print("Video shape:", example_sample['video'].shape)

Sample ID: CLSFBI0301A_S008_B_12100_12437
Label: bonjour
Signer: S008
Nb. of frames: 18
Pose sequence - body parts: {'lips', 'right_hand', 'left_eye', 'left_hand', 'upper_pose', 'right_eye'}
Pose sequence - upper pose shape: (18, 23, 3)
Video shape: torch.Size([18, 3, 480, 600])


In [4]:
def get_sample_per_label(label: str, max_n=10) -> list[str]:
    sample_indices = [i for i, s in enumerate(dataset.samples) if s['label'] == label]
    random.shuffle(sample_indices)
    return [dataset[i] for i in sample_indices[:max_n]]

In [5]:
def inspect_sample(sample):
    player = VideoPlayer()
    player.attach_video_tensor(sample['video'].permute(0, 2, 3, 1), fps=50, name='video')
    # player.attach_empty(800, 600, name='skeleton')
    player.attach_poses(sample['poses']['upper_pose'], UPPER_POSE_EDGES, parent_name='video')
    player.attach_poses(sample['poses']['left_hand'], HAND_EDGES, edge_color=(255, 0, 0), parent_name='video')
    player.attach_poses(sample['poses']['right_hand'], HAND_EDGES, edge_color=(0, 0, 255), parent_name='video')
    player.attach_poses(sample['poses']['lips'], LIPS_EDGES, edge_color=(0, 0, 255), parent_name='video')
    player.attach_poses(sample['poses']['left_eye'], EYE_EDGES, edge_color=(255, 255, 255), parent_name='video')
    player.attach_poses(sample['poses']['right_eye'], EYE_EDGES, edge_color=(255, 255, 255), parent_name='video')
    player.play(speed=0.2)

In [6]:
for sample in get_sample_per_label('rencontrer', max_n=10):
    inspect_sample(sample)